# AH | Perfilado de fuentes

**Autor:** Alan Arturo Hernández Gutiérrez.

Calcula estructura, calidad, relaciones y cobertura de las siete tablas. El cuaderno 01 presenta los resultados y sus decisiones.

Este cuaderno define las operaciones que utilizan los cuadernos de `notebooks/`. Ejecute sus celdas en orden. El flujo completo se inicia desde 01 o 02.

## Raíz del proyecto

La ubicación se resuelve desde la raíz, `src/` o `notebooks/`, sin guardar rutas personales.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {'src', 'notebooks'}:
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'proyecto.json').is_file():
    raise RuntimeError('Inicie Jupyter desde la raíz del proyecto, src/ o notebooks/.')


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
from pathlib import Path

from io import BytesIO

import sys, json, hashlib, platform

import numpy as np

import pandas as pd

import zipfile

OUT = ROOT / "evidencia"

OUT.mkdir(exist_ok=True)

IDCOLS = {
    "TICKET",
    "SKU_CVE",
    "TIENDA_CVE",
    "CUENTA_CVE",
    "CLIENTE_CVE",
    "MUNICIPIO_CVE",
    "ESTADO_CLAVE",
    "MARCA_CVE",
    "CATEGORIA_CVE",
    "SEGMENTO",
}

NUMCOLS = {"EDAD", "IM_2020", "MONTO", "ITEMS", "TIPO_OPERACION"}


## Preparar escalares para el perfil

Convierte valores numéricos de pandas y NumPy a escalares serializables. Un dato ausente se registra como nulo.

In [ ]:
def n(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v


## Normalizar claves de diagnóstico

Compara claves como texto, retirando espacios y el sufijo decimal de una representación de entero. Este paso se utiliza en los controles de relación.

In [ ]:
def norm(s):
    return s.astype("string").str.strip().str.replace("\\.0$", "", regex=True)


## Perfilar una variable

Calcula tipo, nulos, valores distintos y, según el campo, longitudes, rangos, cuantiles o frecuencias. Devuelve la ficha de esa variable.

In [ ]:
def _perfil_campo(s, c):
    field = {
        "dtype": str(s.dtype),
        "nulls": int(s.isna().sum()),
        "distinct_nonnull": int(s.nunique(dropna=True)),
    }
    if c in IDCOLS:
        x = s.dropna().astype(str)
        field["length_counts"] = {
            str(k): int(v) for k, v in x.str.len().value_counts().sort_index().items()
        }
        field["decimal_suffix_rows"] = int(x.str.contains("\\.0$", regex=True).sum())
    if c in NUMCOLS:
        a = pd.to_numeric(s, errors="coerce")
        field.update(
            {
                "min": n(a.min()),
                "max": n(a.max()),
                "negative": int(a.lt(0).sum()),
                "zero": int(a.eq(0).sum()),
                "quantiles": {
                    str(k): n(v) for k, v in a.quantile([0.01, 0.25, 0.5, 0.75, 0.99]).items()
                },
            }
        )
    if c in ["TIPO", "CATEGORIA_NOMBRE", "TIPO_OPERACION", "SEGMENTO"]:
        field["values"] = {str(k): int(v) for k, v in s.value_counts(dropna=False).items()}
    return field


## Leer y perfilar las siete tablas

Lee los CSV del ZIP conservando las claves como texto; registra filas, columnas, hash y duplicados, y agrega la ficha de cada campo.

In [ ]:
def _leer_y_perfilar_tablas(archive, report):
    frames = {}
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if not entry.filename.endswith(".csv"):
                continue
            rawdata = z.read(entry)
            headers = pd.read_csv(BytesIO(rawdata), nrows=0).columns
            d = {
                c: "string"
                for c in headers
                if c in IDCOLS or c in ["FECHA", "NOM_MUN", "TIPO", "CATEGORIA_NOMBRE"]
            }
            df = pd.read_csv(BytesIO(rawdata), dtype=d)
            table = Path(entry.filename).stem
            frames[table] = df
            info = {
                "rows": len(df),
                "columns": len(df.columns),
                "sha256": hashlib.sha256(rawdata).hexdigest(),
                "exact_duplicate_rows": int(df.duplicated().sum()),
                "fields": {},
            }
            info["fields"] = {c: _perfil_campo(df[c], c) for c in df}
            report["tables"][table] = info
            print("PROFILE", table, len(df), flush=True)
    return frames


## Comprobar unicidad de las claves

Registra claves repetidas, exceso de filas y multiplicidad máxima en los catálogos usados para las uniones.

In [ ]:
def _perfil_claves(frames, report):
    for table, key in [
        ("DIM_CATEGORIA", "CATEGORIA_CVE"),
        ("DIM_TIENDA", "TIENDA_CVE"),
        ("DIM_MUNICIPIO", "MUNICIPIO_CVE"),
        ("DIM_SEGMENTOS", "CLIENTE_CVE"),
        ("DIM_CLIENTES", "CUENTA_CVE"),
        ("DIM_PRODUCTO", "SKU_CVE"),
    ]:
        df = frames[table]
        counts = norm(df[key]).value_counts()
        report["tables"][table]["key_diagnostic"] = {
            "key": key,
            "nonnull_unique": int(counts.size),
            "duplicate_key_rows_excess": int((counts - 1).clip(lower=0).sum()),
            "keys_repeated": int(counts.gt(1).sum()),
            "max_rows_per_key": int(counts.max()),
            "duplicate_nonidentical_rows_excess": int(
                len(df.drop_duplicates()) - df[key].nunique(dropna=False)
            ),
        }


## Revisar fechas e identidad del ticket

Comprueba el formato ISO y resume periodo, líneas por mes, importes, unidades y consistencia de fecha, tienda y cuenta por ticket.

In [ ]:
def _perfil_transacciones(frames, report):
    f = frames["FAC_VENTAS"]
    date_iso = f["FECHA"].str.fullmatch("\\d{4}-\\d{2}-\\d{2}", na=False)
    if not date_iso.all():
        raise ValueError("Unexpected source date format; review before interpreting dates")
    f["FECHA_PARSED"] = pd.to_datetime(f["FECHA"], format="%Y-%m-%d", errors="coerce")
    dates = f["FECHA_PARSED"]
    report["transactions"] = {
        "date_min": str(dates.min()),
        "date_max": str(dates.max()),
        "invalid_dates": int(dates.isna().sum()),
        "date_format_samples": list(f["FECHA"].dropna().unique()[:5]),
        "rows_by_month": {
            str(k): int(v) for k, v in dates.dt.to_period("M").value_counts().sort_index().items()
        },
        "money_by_operation": {
            str(k): float(v) for k, v in f.groupby("TIPO_OPERACION")["MONTO"].sum().items()
        },
        "items_by_operation": {
            str(k): float(v) for k, v in f.groupby("TIPO_OPERACION")["ITEMS"].sum().items()
        },
        "ticket_distinct": int(f["TICKET"].nunique()),
        "account_distinct": int(f["CUENTA_CVE"].nunique()),
        "repeated_account_date_rows_excess": int(
            f.duplicated(["CUENTA_CVE", "FECHA_PARSED"]).sum()
        ),
        "ticket_in_multiple_dates": int(f.groupby("TICKET")["FECHA_PARSED"].nunique().gt(1).sum()),
        "ticket_in_multiple_stores": int(f.groupby("TICKET")["TIENDA_CVE"].nunique().gt(1).sum()),
        "ticket_in_multiple_accounts": int(f.groupby("TICKET")["CUENTA_CVE"].nunique().gt(1).sum()),
        "ticket_sku_duplicate_rows_excess": int(f.duplicated(["TICKET", "SKU_CVE"]).sum()),
    }


## Medir cobertura de las uniones

Calcula claves nulas, correspondencias, faltantes y registros que recibirían múltiples coincidencias al unir cada par de fuentes.

In [ ]:
def _perfil_relaciones(frames, report):
    for left, lk, right, rk in [
        ("FAC_VENTAS", "TIENDA_CVE", "DIM_TIENDA", "TIENDA_CVE"),
        ("FAC_VENTAS", "CUENTA_CVE", "DIM_CLIENTES", "CUENTA_CVE"),
        ("FAC_VENTAS", "SKU_CVE", "DIM_PRODUCTO", "SKU_CVE"),
        ("DIM_PRODUCTO", "CATEGORIA_CVE", "DIM_CATEGORIA", "CATEGORIA_CVE"),
        ("DIM_CLIENTES", "MUNICIPIO_CVE", "DIM_MUNICIPIO", "MUNICIPIO_CVE"),
        ("DIM_TIENDA", "MUNICIPIO_CVE", "DIM_MUNICIPIO", "MUNICIPIO_CVE"),
        ("DIM_CLIENTES", "CLIENTE_CVE", "DIM_SEGMENTOS", "CLIENTE_CVE"),
    ]:
        a, b = (frames[left], frames[right])
        keys = norm(b[rk])
        mask = norm(a[lk]).isin(keys.dropna())
        multiplicity = norm(a[lk]).map(keys.value_counts()).fillna(0)
        report["joins"][f"{left}.{lk} -> {right}.{rk}"] = {
            "left_rows": len(a),
            "left_null": int(a[lk].isna().sum()),
            "matched_rows": int(mask.sum()),
            "unmatched_nonnull_rows": int((~mask & a[lk].notna()).sum()),
            "expected_left_join_rows": int(multiplicity.clip(lower=1).sum()),
            "rows_with_multiple_matches": int(multiplicity.gt(1).sum()),
        }


## Describir el catálogo territorial

Resume municipios, nombres repetidos y entidades presentes para interpretar la cobertura geográfica.

In [ ]:
def _perfil_geografia(frames, report):
    geo = frames["DIM_MUNICIPIO"]
    report["geography"] = {
        "samples": geo[["MUNICIPIO_CVE", "NOM_MUN", "ESTADO_CLAVE", "IM_2020"]]
        .head(12)
        .to_dict("records"),
        "states": {
            str(k): int(v) for k, v in geo["ESTADO_CLAVE"].value_counts().sort_index().items()
        },
        "name_state_duplicates": int(geo.duplicated(["ESTADO_CLAVE", "NOM_MUN"]).sum()),
        "same_name_multiple_states": int(
            geo.groupby("NOM_MUN")["ESTADO_CLAVE"].nunique().gt(1).sum()
        ),
        "municipality_key_equal_concat_state_suffix": None,
    }


## Revisar cuentas y atributos de cliente

Calcula cuentas por cliente y comprueba discrepancias de edad, municipio e identidad entre cuentas.

In [ ]:
def _perfil_clientes(frames, report):
    clients = frames["DIM_CLIENTES"]
    report["customers"] = {
        "customer_distinct": int(clients["CLIENTE_CVE"].nunique()),
        "accounts_per_customer_max": int(
            clients.groupby("CLIENTE_CVE")["CUENTA_CVE"].nunique().max()
        ),
        "customers_multiple_accounts": int(
            clients.groupby("CLIENTE_CVE")["CUENTA_CVE"].nunique().gt(1).sum()
        ),
        "account_customer_conflicts": int(
            clients.groupby("CUENTA_CVE")["CLIENTE_CVE"].nunique().gt(1).sum()
        ),
        "ages_under_18": int(clients["EDAD"].lt(18).sum()),
        "ages_over_100": int(clients["EDAD"].gt(100).sum()),
        "municipalities_per_customer_gt1": int(
            clients.groupby("CLIENTE_CVE")["MUNICIPIO_CVE"].nunique().gt(1).sum()
        ),
    }


## Contrastar el primer día observado

Solo después de comprobar claves únicas, une movimientos con cliente y canal. Clasifica los canales del primer día sin inventar una secuencia intradía.

In [ ]:
def _perfil_canales(frames, report):
    f = frames["FAC_VENTAS"]
    clients = frames["DIM_CLIENTES"]
    if (
        report["tables"]["DIM_CLIENTES"]["key_diagnostic"]["keys_repeated"] == 0
        and report["tables"]["DIM_TIENDA"]["key_diagnostic"]["keys_repeated"] == 0
    ):
        tx = (
            f[["CUENTA_CVE", "TIENDA_CVE", "FECHA_PARSED", "TIPO_OPERACION", "MONTO", "ITEMS"]]
            .merge(
                clients[["CUENTA_CVE", "CLIENTE_CVE"]],
                on="CUENTA_CVE",
                how="left",
                validate="many_to_one",
            )
            .merge(
                frames["DIM_TIENDA"][["TIENDA_CVE", "TIPO"]],
                on="TIENDA_CVE",
                how="left",
                validate="many_to_one",
            )
        )
        pur = tx.loc[tx["TIPO_OPERACION"].eq(1) & tx["CLIENTE_CVE"].notna()]
        channels = (
            pur.groupby("TIPO", dropna=False)
            .agg(
                rows=("MONTO", "size"),
                amount=("MONTO", "sum"),
                customers=("CLIENTE_CVE", "nunique"),
            )
            .reset_index()
        )
        report["purchase_diagnostic"] = {
            "matched_purchase_rows": len(pur),
            "channels": channels.to_dict("records"),
        }
        first = pur.groupby("CLIENTE_CVE")["FECHA_PARSED"].min()
        first_rows = pur.loc[pur["FECHA_PARSED"].eq(pur["CLIENTE_CVE"].map(first))]
        flags = (
            first_rows.assign(
                is_online=first_rows["TIPO"].eq("Internet"),
                is_physical=first_rows["TIPO"].eq("Fisica"),
                is_boutique=first_rows["TIPO"].eq("Boutique"),
                is_unknown=first_rows["TIPO"].isna(),
            )
            .groupby("CLIENTE_CVE")[["is_online", "is_physical", "is_boutique", "is_unknown"]]
            .max()
        )
        report["purchase_diagnostic"]["first_day_customer_flags"] = {
            "customers": len(flags),
            "online_only": int(
                (
                    flags.is_online & ~flags.is_physical & ~flags.is_boutique & ~flags.is_unknown
                ).sum()
            ),
            "online_and_physical": int((flags.is_online & flags.is_physical).sum()),
            "physical_without_online": int(
                (flags.is_physical & ~flags.is_online & ~flags.is_unknown).sum()
            ),
            "boutique_only": int(
                (
                    flags.is_boutique & ~flags.is_online & ~flags.is_physical & ~flags.is_unknown
                ).sum()
            ),
            "unknown_channel": int(flags.is_unknown.sum()),
        }


## Perfilar fuentes

Organiza los perfiles de las siete tablas y guarda sus comprobaciones agregadas.

In [ ]:
def perfilar_fuentes(archive):
    report = {
        "environment": {"python": platform.python_version(), "pandas": pd.__version__},
        "source": {"archive_sha256": hashlib.sha256(Path(archive).read_bytes()).hexdigest()},
        "tables": {},
        "joins": {},
    }
    frames = _leer_y_perfilar_tablas(archive, report)
    _perfil_claves(frames, report)
    _perfil_transacciones(frames, report)
    _perfil_relaciones(frames, report)
    _perfil_geografia(frames, report)
    _perfil_clientes(frames, report)
    _perfil_canales(frames, report)
    path = OUT / "perfil_fuentes.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=n), encoding="utf-8")
    print("SAVED", path.name)
    print(
        json.dumps(
            {k: v for k, v in report.items() if k not in ["tables"]},
            ensure_ascii=False,
            indent=2,
            default=n,
        )
    )
    return report
